# Orbit Wars JAX PPO Train

Notebook nay la huong JAX dung: khong train tren grid, khong dung PyTorch rollout cham.

Muc tieu:
1. Lay official engine lam ground truth.
2. Port simulator sang JAX voi state/action tensor co kich thuoc co dinh.
3. Validate JAX step voi official engine tren cac case nho.
4. Sau khi validate moi train PPO vectorized.

Luu y quan trong: neu JAX simulator chua validate pass thi KHONG nen tin ket qua PPO.


In [ ]:
# === JAX TRAIN CONFIG ===
# Cell nay la noi ban sua config chinh.

from pathlib import Path

# Kich thuoc tensor co dinh. Game that thuong 20-40 planets, 256 fleets la du rong cho rollout.
MAX_PLANETS = 64
MAX_FLEETS = 256
MAX_MOVES_PER_PLAYER = 8
MAX_PLAYERS = 4
MAX_COMET_GROUPS = 5
MAX_COMET_PATH = 40
COMET_SPAWN_STEPS = (50, 150, 250, 350, 450)

# Train 2P + 4P. Neu chi muon 4P, dat TRAIN_PROB_4P = 1.0.
TRAIN_PROB_4P = 0.75
TRAIN_PROB_2P = 0.25

# So env song song va so step moi rollout. Tong samples moi update = NUM_ENVS * ROLLOUT_STEPS.
NUM_ENVS = 512
ROLLOUT_STEPS = 128
TOTAL_PPO_UPDATES = 500

# Eval fixed sau moi N update.
EVAL_EVERY_UPDATES = 25
EVAL_4P_MATCHES = 64
EVAL_2P_MATCHES = 32

# PPO params.
GAMMA = 0.995
GAE_LAMBDA = 0.95
PPO_CLIP_EPS = 0.20
ENTROPY_COEF = 0.01
VALUE_COEF = 0.50
LEARNING_RATE = 3e-4
UPDATE_EPOCHS = 4
MINIBATCH_SIZE = 4096

# Candidate-bias policy. Heuristic van sinh candidate/action hop le.
FEATURE_DIM = 18
NUM_CANDIDATES = 32
HIDDEN_DIM = 64
BIAS_SCALE = 2.0
TRAIN_TEMPERATURE = 1.25

# Notebook run control.
# AUTO_RUN_VALIDATION=True: Run all se tu check env va in PASS/FAIL.
# AUTO_START_TRAIN=False: Run all khong tu train dai. Doi True khi validation da pass va ban muon train.
AUTO_RUN_VALIDATION = True
AUTO_START_TRAIN = False

# JAX simulator validation.
VALIDATE_RANDOM_SEEDS = [1, 2, 3, 4, 5]
VALIDATE_STEPS = 30
VALIDATE_WITH_COMETS = False  # Bat True sau khi core no-comet da pass.

# Official opponent paths chi dung cho eval / imitation data neu can.
TRAIN_OPPONENT_POOL_4P = [
    Path('/kaggle/input/YOUR_DATASET_1/agent_a.py'),
    Path('/kaggle/input/YOUR_DATASET_2/agent_b.py'),
    Path('/kaggle/input/YOUR_DATASET_3/agent_c.py'),
    Path('/kaggle/input/YOUR_DATASET_4/agent_d.py'),
    Path('/kaggle/input/YOUR_DATASET_5/agent_e.py'),
    Path('/kaggle/input/YOUR_DATASET_6/agent_f.py'),
    Path('/kaggle/input/YOUR_DATASET_7/agent_g.py'),
]

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = WORK_DIR / 'jax_ppo_out'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('NUM_ENVS', NUM_ENVS)
print('ROLLOUT_STEPS', ROLLOUT_STEPS)
print('TOTAL_PPO_UPDATES', TOTAL_PPO_UPDATES)
print('samples/update', NUM_ENVS * ROLLOUT_STEPS)
print('OUT_DIR', OUT_DIR)


In [ ]:
# === IMPORTS AND OFFICIAL ENGINE ===
# Doc official engine de dung lam ground truth va reset/map generator.

import math
import random
import json
from dataclasses import dataclass
from typing import NamedTuple

import numpy as np
import jax
import jax.numpy as jnp
from jax import lax, random as jrandom

from kaggle_environments import make
from kaggle_environments.envs.orbit_wars import orbit_wars as ow

BOARD_SIZE = float(ow.BOARD_SIZE)
CENTER = float(ow.CENTER)
SUN_RADIUS = float(ow.SUN_RADIUS)
ROTATION_RADIUS_LIMIT = float(ow.ROTATION_RADIUS_LIMIT)
COMET_RADIUS = float(ow.COMET_RADIUS)
COMET_PRODUCTION = float(ow.COMET_PRODUCTION)
COMET_SPAWN_STEPS = tuple(int(x) for x in ow.COMET_SPAWN_STEPS)

print('official orbit_wars loaded')
print('BOARD_SIZE', BOARD_SIZE, 'SUN_RADIUS', SUN_RADIUS, 'ROT_LIMIT', ROTATION_RADIUS_LIMIT)


In [ ]:
# === JAX STATE / ACTION FORMAT ===
# planets: [B, P, 7] = id, owner, x, y, radius, ships, production. Dead slots id=-1.
# fleets:  [B, F, 7] = id, owner, x, y, angle, from_planet_id, ships. Dead slots id=-1.
# actions: [B, A, M, 3] = from_planet_id, angle, ships. Invalid moves ships<=0.
# comet_paths: [B, G, 4, T, 2] precomputed by official generator from hidden seed.

class JaxState(NamedTuple):
    planets: jnp.ndarray
    fleets: jnp.ndarray
    initial_planets: jnp.ndarray
    comet_paths: jnp.ndarray
    comet_lengths: jnp.ndarray
    comet_ships: jnp.ndarray
    comet_base_id: jnp.ndarray
    step: jnp.ndarray
    next_fleet_id: jnp.ndarray
    angular_velocity: jnp.ndarray
    player_count: jnp.ndarray
    done: jnp.ndarray
    rewards: jnp.ndarray

class JaxAction(NamedTuple):
    moves: jnp.ndarray


def empty_state(batch_size: int, player_count: int = 4):
    planets = jnp.full((batch_size, MAX_PLANETS, 7), -1.0, dtype=jnp.float32)
    fleets = jnp.full((batch_size, MAX_FLEETS, 7), -1.0, dtype=jnp.float32)
    return JaxState(
        planets=planets,
        fleets=fleets,
        initial_planets=planets,
        comet_paths=jnp.zeros((batch_size, MAX_COMET_GROUPS, 4, MAX_COMET_PATH, 2), dtype=jnp.float32),
        comet_lengths=jnp.zeros((batch_size, MAX_COMET_GROUPS, 4), dtype=jnp.int32),
        comet_ships=jnp.zeros((batch_size, MAX_COMET_GROUPS), dtype=jnp.float32),
        comet_base_id=jnp.zeros((batch_size,), dtype=jnp.int32),
        step=jnp.zeros((batch_size,), dtype=jnp.int32),
        next_fleet_id=jnp.zeros((batch_size,), dtype=jnp.int32),
        angular_velocity=jnp.zeros((batch_size,), dtype=jnp.float32),
        player_count=jnp.full((batch_size,), int(player_count), dtype=jnp.int32),
        done=jnp.zeros((batch_size,), dtype=jnp.bool_),
        rewards=jnp.zeros((batch_size, MAX_PLAYERS), dtype=jnp.float32),
    )


def pad_rows(rows, max_rows, width=7, fill=-1.0):
    arr = np.full((max_rows, width), fill, dtype=np.float32)
    rows = np.asarray(rows, dtype=np.float32)
    n = min(len(rows), max_rows)
    if n:
        arr[:n] = rows[:n]
    return arr


In [ ]:
# === JAX GEOMETRY MATCHING OFFICIAL ENGINE ===
# Port cac ham geometry tu orbit_wars.py sang JAX.

@jax.jit
def fleet_speed(ships, max_speed=6.0):
    ships = jnp.maximum(ships, 1.0)
    speed = 1.0 + (max_speed - 1.0) * (jnp.log(ships) / jnp.log(1000.0)) ** 1.5
    return jnp.minimum(speed, max_speed)

@jax.jit
def point_to_segment_distance(px, py, vx, vy, wx, wy):
    l2 = (vx - wx) ** 2 + (vy - wy) ** 2
    def nonzero(_):
        t = ((px - vx) * (wx - vx) + (py - vy) * (wy - vy)) / l2
        t = jnp.clip(t, 0.0, 1.0)
        projx = vx + t * (wx - vx)
        projy = vy + t * (wy - vy)
        return jnp.sqrt((px - projx) ** 2 + (py - projy) ** 2)
    def zero(_):
        return jnp.sqrt((px - vx) ** 2 + (py - vy) ** 2)
    return lax.cond(l2 == 0.0, zero, nonzero, operand=None)

@jax.jit
def swept_pair_hit(ax, ay, bx, by, p0x, p0y, p1x, p1y, radius):
    d0x = ax - p0x
    d0y = ay - p0y
    dvx = (bx - ax) - (p1x - p0x)
    dvy = (by - ay) - (p1y - p0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - radius * radius
    disc = b * b - 4.0 * a * c
    def small_a(_):
        return c <= 0.0
    def normal(_):
        sq = jnp.sqrt(jnp.maximum(disc, 0.0))
        t1 = (-b - sq) / (2.0 * a)
        t2 = (-b + sq) / (2.0 * a)
        return (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    return lax.cond(a < 1e-12, small_a, normal, operand=None)


In [ ]:
# === OFFICIAL RESET TO JAX STATE ===
# Dung official engine de tao initial state va pack vao tensor JAX.
# Comet path/ships duoc precompute bang official generator theo hidden seed.

from types import SimpleNamespace


def official_initial_state(num_agents=4, seed=0, episode_steps=500, ship_speed=6.0, comet_speed=4.0):
    state = [
        SimpleNamespace(observation=SimpleNamespace(step=0), action=[], status='ACTIVE', reward=0)
    ] + [
        SimpleNamespace(observation=SimpleNamespace(player=i), action=[], status='ACTIVE', reward=0)
        for i in range(1, num_agents)
    ]
    env = SimpleNamespace(
        configuration=SimpleNamespace(seed=seed, shipSpeed=ship_speed, episodeSteps=episode_steps, cometSpeed=comet_speed),
        done=False,
        info={},
    )
    state = ow.interpreter(state, env)
    # Keep seed only inside this training notebook so schedule can be precomputed.
    state[0].observation.episode_seed = env.info.get('seed', seed)
    return state, env


def precompute_comet_schedule(initial_planets, angular_velocity, episode_seed, comet_speed=4.0):
    paths_arr = np.zeros((MAX_COMET_GROUPS, 4, MAX_COMET_PATH, 2), dtype=np.float32)
    lengths = np.zeros((MAX_COMET_GROUPS, 4), dtype=np.int32)
    ships = np.zeros((MAX_COMET_GROUPS,), dtype=np.float32)
    comet_planet_ids = []
    for gi, spawn_step in enumerate(COMET_SPAWN_STEPS[:MAX_COMET_GROUPS]):
        rng = random.Random(f'orbit_wars-comet-{int(episode_seed)}-{int(spawn_step)}')
        paths = ow.generate_comet_paths(
            initial_planets,
            float(angular_velocity),
            int(spawn_step),
            comet_planet_ids,
            float(comet_speed),
            rng=rng,
        )
        if paths:
            ships[gi] = float(min(rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99)))
            for ci, path in enumerate(paths[:4]):
                n = min(len(path), MAX_COMET_PATH)
                lengths[gi, ci] = n
                if n:
                    paths_arr[gi, ci, :n] = np.asarray(path[:n], dtype=np.float32)
            # Official comet lifetimes are <= 40 and spawn windows are 100 turns apart,
            # so no prior comet remains active at the next spawn in default settings.
            comet_planet_ids = []
    return paths_arr, lengths, ships


def pack_official_states(initial_states, comet_speed=4.0):
    planets = []
    fleets = []
    initials = []
    comet_paths = []
    comet_lengths = []
    comet_ships = []
    comet_base_ids = []
    steps = []
    next_ids = []
    ang = []
    rewards = []
    done = []
    player_counts = []
    for state in initial_states:
        obs = state[0].observation
        planets_np = pad_rows(obs.planets, MAX_PLANETS)
        initials_np = pad_rows(obs.initial_planets, MAX_PLANETS)
        live_initial = [p for p in obs.initial_planets if p[0] >= 0]
        base_id = int(max(p[0] for p in live_initial) + 1) if live_initial else 0
        seed = int(getattr(obs, 'episode_seed', 0))
        paths_np, lengths_np, ships_np = precompute_comet_schedule(
            [list(p) for p in obs.initial_planets],
            float(getattr(obs, 'angular_velocity', 0.0)),
            seed,
            comet_speed=comet_speed,
        )
        planets.append(planets_np)
        fleets.append(pad_rows(getattr(obs, 'fleets', []), MAX_FLEETS))
        initials.append(initials_np)
        comet_paths.append(paths_np)
        comet_lengths.append(lengths_np)
        comet_ships.append(ships_np)
        comet_base_ids.append(base_id)
        steps.append(int(getattr(obs, 'step', 0)))
        next_ids.append(int(getattr(obs, 'next_fleet_id', 0)))
        ang.append(float(getattr(obs, 'angular_velocity', 0.0)))
        rewards.append([float(getattr(s, 'reward', 0.0)) for s in state] + [0.0] * (MAX_PLAYERS - len(state)))
        done.append(all(getattr(s, 'status', 'ACTIVE') == 'DONE' for s in state))
        player_counts.append(len(state))
    return JaxState(
        planets=jnp.asarray(np.stack(planets), dtype=jnp.float32),
        fleets=jnp.asarray(np.stack(fleets), dtype=jnp.float32),
        initial_planets=jnp.asarray(np.stack(initials), dtype=jnp.float32),
        comet_paths=jnp.asarray(np.stack(comet_paths), dtype=jnp.float32),
        comet_lengths=jnp.asarray(np.stack(comet_lengths), dtype=jnp.int32),
        comet_ships=jnp.asarray(np.stack(comet_ships), dtype=jnp.float32),
        comet_base_id=jnp.asarray(comet_base_ids, dtype=jnp.int32),
        step=jnp.asarray(steps, dtype=jnp.int32),
        next_fleet_id=jnp.asarray(next_ids, dtype=jnp.int32),
        angular_velocity=jnp.asarray(ang, dtype=jnp.float32),
        player_count=jnp.asarray(player_counts, dtype=jnp.int32),
        done=jnp.asarray(done, dtype=jnp.bool_),
        rewards=jnp.asarray(np.asarray(rewards, dtype=np.float32)),
    )

# Smoke reset.
states = [official_initial_state(num_agents=4, seed=s)[0] for s in [1, 2]]
jstate = pack_official_states(states)
print(jstate.planets.shape, jstate.fleets.shape, jstate.step, jstate.comet_paths.shape)


In [ ]:
# === JAX STEP CORE ===
# Core JAX simulator ported from official orbit_wars.py.
# Supported: launch, production, orbiting planets, comet schedule, fleet movement/collision, combat, reward.


def _set_row(arr, idx, row):
    return arr.at[idx].set(row)


def _process_one_move(carry, move_data):
    planets, fleets, next_fleet_id, player_id, player_count = carry
    move = move_data
    from_id = move[0].astype(jnp.int32)
    angle = move[1]
    ships = move[2].astype(jnp.int32).astype(jnp.float32)

    p_ids = planets[:, 0].astype(jnp.int32)
    p_alive = p_ids >= 0
    source_mask = p_alive & (p_ids == from_id)
    p_idx = jnp.argmax(source_mask.astype(jnp.int32))
    source_exists = source_mask[p_idx]
    owned = planets[p_idx, 1].astype(jnp.int32) == player_id
    enough = planets[p_idx, 5] >= ships
    valid_player = player_id < player_count
    valid = source_exists & owned & enough & (ships > 0.0) & valid_player

    f_alive = fleets[:, 0] >= 0
    free_mask = ~f_alive
    f_idx = jnp.argmax(free_mask.astype(jnp.int32))
    free_exists = free_mask[f_idx]
    valid = valid & free_exists

    radius = planets[p_idx, 4]
    start_x = planets[p_idx, 2] + jnp.cos(angle) * (radius + 0.1)
    start_y = planets[p_idx, 3] + jnp.sin(angle) * (radius + 0.1)
    new_fleet = jnp.array([
        next_fleet_id.astype(jnp.float32),
        player_id.astype(jnp.float32),
        start_x,
        start_y,
        angle,
        from_id.astype(jnp.float32),
        ships,
    ], dtype=jnp.float32)

    planets2 = lax.cond(
        valid,
        lambda x: x.at[p_idx, 5].add(-ships),
        lambda x: x,
        planets,
    )
    fleets2 = lax.cond(valid, lambda x: x.at[f_idx].set(new_fleet), lambda x: x, fleets)
    next2 = next_fleet_id + valid.astype(jnp.int32)
    return (planets2, fleets2, next2, player_id, player_count), None


def _process_player_moves(carry, player_id):
    planets, fleets, next_fleet_id, actions, player_count = carry
    moves = actions[player_id]
    (planets, fleets, next_fleet_id, _, _), _ = lax.scan(
        _process_one_move,
        (planets, fleets, next_fleet_id, player_id, player_count),
        moves,
    )
    return (planets, fleets, next_fleet_id, actions, player_count), None


def process_launches_single(planets, fleets, next_fleet_id, actions, player_count):
    (planets, fleets, next_fleet_id, _, _), _ = lax.scan(
        _process_player_moves,
        (planets, fleets, next_fleet_id, actions, player_count),
        jnp.arange(MAX_PLAYERS, dtype=jnp.int32),
    )
    return planets, fleets, next_fleet_id


def production_single(planets):
    alive = planets[:, 0] >= 0
    owned = planets[:, 1] != -1
    add = jnp.where(alive & owned, planets[:, 6], 0.0)
    return planets.at[:, 5].add(add)


def _spawn_one_comet_copy(carry, copy_idx):
    planets, initial_planets, group_idx, comet_ships_value, comet_base_id, should_spawn = carry
    free_mask = planets[:, 0] < 0
    p_idx = jnp.argmax(free_mask.astype(jnp.int32))
    free_exists = free_mask[p_idx]
    valid = should_spawn & free_exists
    pid = comet_base_id + copy_idx
    row = jnp.array([
        pid.astype(jnp.float32),
        -1.0,
        -99.0,
        -99.0,
        float(ow.COMET_RADIUS),
        comet_ships_value,
        float(ow.COMET_PRODUCTION),
    ], dtype=jnp.float32)
    planets = lax.cond(valid, lambda x: x.at[p_idx].set(row), lambda x: x, planets)
    initial_planets = lax.cond(valid, lambda x: x.at[p_idx].set(row), lambda x: x, initial_planets)
    return (planets, initial_planets, group_idx, comet_ships_value, comet_base_id, should_spawn), None


def spawn_comets_single(planets, initial_planets, step, comet_lengths, comet_ships, comet_base_id):
    spawn_steps = jnp.asarray(COMET_SPAWN_STEPS, dtype=jnp.int32)
    next_step = step + jnp.asarray(1, dtype=jnp.int32)
    spawn_mask = spawn_steps == next_step
    group_idx = jnp.argmax(spawn_mask.astype(jnp.int32))
    has_spawn_step = jnp.any(spawn_mask)
    has_path = comet_lengths[group_idx, 0] > 0
    should_spawn = has_spawn_step & has_path
    comet_ships_value = comet_ships[group_idx]
    (planets, initial_planets, *_), _ = lax.scan(
        _spawn_one_comet_copy,
        (planets, initial_planets, group_idx, comet_ships_value, comet_base_id, should_spawn),
        jnp.arange(4, dtype=jnp.int32),
    )
    return planets, initial_planets


def compute_planet_paths_single(planets, initial_planets, step, angular_velocity, comet_paths, comet_lengths, comet_base_id):
    old_xy = planets[:, 2:4]
    init_xy = initial_planets[:, 2:4]
    p_ids = planets[:, 0].astype(jnp.int32)
    comet_copy = p_ids - comet_base_id
    is_comet = (planets[:, 0] >= 0) & (comet_copy >= 0) & (comet_copy < 4)

    dx = init_xy[:, 0] - CENTER
    dy = init_xy[:, 1] - CENTER
    orb_r = jnp.sqrt(dx * dx + dy * dy)
    init_angle = jnp.arctan2(dy, dx)
    radius = planets[:, 4]
    alive = planets[:, 0] >= 0
    has_init = initial_planets[:, 0] >= 0
    orbiting = alive & has_init & (~is_comet) & ((orb_r + radius) < ROTATION_RADIUS_LIMIT)
    cur_angle = init_angle + angular_velocity * step.astype(jnp.float32)
    orbit_x = CENTER + orb_r * jnp.cos(cur_angle)
    orbit_y = CENTER + orb_r * jnp.sin(cur_angle)
    orbit_xy = jnp.stack([orbit_x, orbit_y], axis=-1)

    spawn_steps = jnp.asarray(COMET_SPAWN_STEPS, dtype=jnp.int32)
    path_idx_by_group = (step + jnp.asarray(1, dtype=jnp.int32)) - spawn_steps
    active_group_mask = path_idx_by_group >= 0
    group_idx = jnp.argmax(jnp.where(active_group_mask, jnp.arange(MAX_COMET_GROUPS, dtype=jnp.int32), -1))
    path_idx = path_idx_by_group[group_idx]
    safe_copy = jnp.clip(comet_copy, 0, 3)
    safe_path_idx = jnp.clip(path_idx, 0, MAX_COMET_PATH - 1)
    comet_xy_all = comet_paths[group_idx, safe_copy, safe_path_idx]
    comet_len = comet_lengths[group_idx, safe_copy]
    comet_expired = is_comet & (path_idx >= comet_len)
    comet_has_path = is_comet & (comet_len > 0) & (path_idx >= 0)
    comet_new_xy = jnp.where(comet_has_path[:, None] & (~comet_expired)[:, None], comet_xy_all, old_xy)

    new_xy = jnp.where(orbiting[:, None], orbit_xy, old_xy)
    new_xy = jnp.where(is_comet[:, None], comet_new_xy, new_xy)
    comet_first_placement = is_comet & (old_xy[:, 0] < 0.0)
    check = alive & (~comet_first_placement)
    return old_xy, new_xy, check, comet_expired

def _move_one_fleet(carry, f_idx):
    fleets, combat, planets, p_old, p_new, p_check, ship_speed = carry
    fleet = fleets[f_idx]
    alive = fleet[0] >= 0
    owner = fleet[1].astype(jnp.int32)
    angle = fleet[4]
    ships = fleet[6]
    spd = fleet_speed(ships, ship_speed)
    old_x = fleet[2]
    old_y = fleet[3]
    new_x = old_x + jnp.cos(angle) * spd
    new_y = old_y + jnp.sin(angle) * spd

    hit_mask = jax.vmap(lambda ox, oy, nx, ny, r, chk: chk & swept_pair_hit(old_x, old_y, new_x, new_y, ox, oy, nx, ny, r))(
        p_old[:, 0], p_old[:, 1], p_new[:, 0], p_new[:, 1], planets[:, 4], p_check
    )
    hit_any = alive & jnp.any(hit_mask)
    hit_idx = jnp.argmax(hit_mask.astype(jnp.int32))
    in_bounds = (0.0 <= new_x) & (new_x <= BOARD_SIZE) & (0.0 <= new_y) & (new_y <= BOARD_SIZE)
    sun_dist = point_to_segment_distance(CENTER, CENTER, old_x, old_y, new_x, new_y)
    hit_sun = sun_dist < SUN_RADIUS
    remove = hit_any | (alive & (~in_bounds)) | (alive & (~hit_any) & hit_sun)

    fleet_moved = fleet.at[2].set(new_x).at[3].set(new_y)
    dead_fleet = jnp.full_like(fleet, -1.0)
    updated_fleet = jnp.where(remove, dead_fleet, fleet_moved)
    updated_fleet = jnp.where(alive, updated_fleet, fleet)
    fleets = fleets.at[f_idx].set(updated_fleet)

    valid_owner = (owner >= 0) & (owner < MAX_PLAYERS)
    combat = lax.cond(
        hit_any & valid_owner,
        lambda c: c.at[hit_idx, owner].add(ships),
        lambda c: c,
        combat,
    )
    return (fleets, combat, planets, p_old, p_new, p_check, ship_speed), None


def move_fleets_single(fleets, planets, p_old, p_new, p_check, ship_speed=6.0):
    combat = jnp.zeros((MAX_PLANETS, MAX_PLAYERS), dtype=jnp.float32)
    (fleets, combat, *_), _ = lax.scan(
        _move_one_fleet,
        (fleets, combat, planets, p_old, p_new, p_check, jnp.asarray(ship_speed, dtype=jnp.float32)),
        jnp.arange(MAX_FLEETS, dtype=jnp.int32),
    )
    return fleets, combat


def apply_planet_movement_single(planets, p_new):
    return planets.at[:, 2:4].set(p_new)


def _combat_one_planet(planets, args):
    p_idx, ships_by_owner = args
    planet = planets[p_idx]
    alive = planet[0] >= 0
    total_attack = ships_by_owner.sum()
    top_owner = jnp.argmax(ships_by_owner).astype(jnp.int32)
    top_ships = ships_by_owner[top_owner]
    top_count = jnp.sum((ships_by_owner == top_ships) & (ships_by_owner > 0.0))
    second_ships = jnp.max(jnp.where(jnp.arange(MAX_PLAYERS) == top_owner, -1.0, ships_by_owner))
    survivor = jnp.where(top_count > 1, 0.0, top_ships - jnp.maximum(second_ships, 0.0))
    has_survivor = alive & (total_attack > 0.0) & (survivor > 0.0)
    same_owner = planet[1].astype(jnp.int32) == top_owner
    reinforced = planet.at[5].add(survivor)
    attacked_ships = planet[5] - survivor
    captured = attacked_ships < 0.0
    attacked = planet.at[5].set(jnp.where(captured, -attacked_ships, attacked_ships)).at[1].set(jnp.where(captured, top_owner.astype(jnp.float32), planet[1]))
    new_planet = lax.cond(same_owner, lambda _: reinforced, lambda _: attacked, operand=None)
    planets = planets.at[p_idx].set(jnp.where(has_survivor, new_planet, planet))
    return planets, None


def combat_single(planets, combat):
    args = (jnp.arange(MAX_PLANETS, dtype=jnp.int32), combat)
    planets, _ = lax.scan(_combat_one_planet, planets, args)
    return planets


def termination_reward_single(planets, fleets, step, player_count, episode_steps=500):
    p_alive = planets[:, 0] >= 0
    f_alive = fleets[:, 0] >= 0
    p_owner = planets[:, 1].astype(jnp.int32)
    f_owner = fleets[:, 1].astype(jnp.int32)
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    active_by_planet = jax.vmap(lambda pl: jnp.any(p_alive & (p_owner == pl)))(players)
    active_by_fleet = jax.vmap(lambda pl: jnp.any(f_alive & (f_owner == pl)))(players)
    valid_player = players < player_count
    alive_players = (active_by_planet | active_by_fleet) & valid_player
    terminated = (step >= (episode_steps - 2)) | (jnp.sum(alive_players.astype(jnp.int32)) <= 1)

    planet_scores = jax.vmap(lambda pl: jnp.sum(jnp.where(p_alive & (p_owner == pl), planets[:, 5], 0.0)))(players)
    fleet_scores = jax.vmap(lambda pl: jnp.sum(jnp.where(f_alive & (f_owner == pl), fleets[:, 6], 0.0)))(players)
    scores = planet_scores + fleet_scores
    max_score = jnp.max(jnp.where(valid_player, scores, -1.0))
    rewards = jnp.where((scores == max_score) & (max_score > 0.0) & valid_player, 1.0, -1.0)
    rewards = jnp.where(terminated, rewards, jnp.zeros_like(rewards))
    return terminated, rewards


def remove_expired_comets_single(planets, initial_planets, comet_expired):
    dead_planet = jnp.full((7,), -1.0, dtype=jnp.float32)
    planets = jnp.where(comet_expired[:, None], dead_planet[None, :], planets)
    initial_planets = jnp.where(comet_expired[:, None], dead_planet[None, :], initial_planets)
    return planets, initial_planets


def jax_step_single(state: JaxState, action: JaxAction, ship_speed=6.0, episode_steps=500):
    planets, initial_planets = spawn_comets_single(
        state.planets,
        state.initial_planets,
        state.step,
        state.comet_lengths,
        state.comet_ships,
        state.comet_base_id,
    )
    planets, fleets, next_id = process_launches_single(planets, state.fleets, state.next_fleet_id, action.moves, state.player_count)
    planets = production_single(planets)
    p_old, p_new, p_check, comet_expired = compute_planet_paths_single(
        planets,
        initial_planets,
        state.step,
        state.angular_velocity,
        state.comet_paths,
        state.comet_lengths,
        state.comet_base_id,
    )
    fleets, combat = move_fleets_single(fleets, planets, p_old, p_new, p_check, ship_speed=ship_speed)
    planets = apply_planet_movement_single(planets, p_new)
    planets, initial_planets = remove_expired_comets_single(planets, initial_planets, comet_expired)
    planets = combat_single(planets, combat)
    done, rewards = termination_reward_single(planets, fleets, state.step, state.player_count, episode_steps=episode_steps)
    return JaxState(
        planets=planets,
        fleets=fleets,
        initial_planets=initial_planets,
        comet_paths=state.comet_paths,
        comet_lengths=state.comet_lengths,
        comet_ships=state.comet_ships,
        comet_base_id=state.comet_base_id,
        step=state.step + jnp.asarray(1, dtype=jnp.int32),
        next_fleet_id=next_id,
        angular_velocity=state.angular_velocity,
        player_count=state.player_count,
        done=done,
        rewards=rewards,
    )


jax_step = jax.jit(jax.vmap(jax_step_single, in_axes=(0, 0, None, None)))
print('jax_step core with comet schedule ready')


In [ ]:
# === VALIDATION HARNESS DESIGN ===
# So sanh official engine va JAX engine tren cung initial state/action sequence.
# Khi port xong jax_step, chay ham nay. Neu fail thi khong train PPO.


def compare_planets_np(a, b, atol=1e-5):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.shape != b.shape:
        return False, f'shape mismatch {a.shape} vs {b.shape}'
    diff = np.nanmax(np.abs(a - b)) if a.size else 0.0
    return diff <= atol, f'max_abs_diff={diff}'


def validate_jax_step_against_official(jax_step_fn, seeds=VALIDATE_RANDOM_SEEDS, steps=VALIDATE_STEPS, num_agents=4):
    # Action empty de validate production/rotation/movement no-action.
    # Sau do them scripted launches de validate launch/combat.
    for seed in seeds:
        official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
        js = pack_official_states([official_state])
        for t in range(steps):
            for s in official_state:
                s.action = []
            official_state = ow.interpreter(official_state, official_env)
            # Kaggle wrapper thuong tang step ben ngoai interpreter; test local can tang tay.
            next_step = getattr(official_state[0].observation, 'step', 0) + 1
            for s in official_state:
                s.observation.step = next_step
            empty_actions = JaxAction(moves=jnp.zeros((1, MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32))
            js = jax_step_fn(js, empty_actions, 6.0, 500)
            jp = np.asarray(js.planets[0])
            op = pad_rows(official_state[0].observation.planets, MAX_PLANETS)
            ok, msg = compare_planets_np(jp, op, atol=1e-4)
            if not ok:
                print('FAIL seed', seed, 't', t, msg)
                return False
    print('validation passed')
    return True

# Chay sau cell jax_step:
# validate_jax_step_against_official(jax_step, seeds=[1], steps=10, num_agents=4)


def pack_actions_for_jax(action_lists, batch_size=1):
    arr = np.zeros((batch_size, MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    for b, per_player in enumerate(action_lists):
        for player_id, moves in enumerate(per_player):
            for m, move in enumerate(moves[:MAX_MOVES_PER_PLAYER]):
                arr[b, player_id, :] = arr[b, player_id, :]  # keep shape materialized
                arr[b, player_id, m] = np.asarray(move, dtype=np.float32)
    return JaxAction(moves=jnp.asarray(arr, dtype=jnp.float32))


def compare_fleets_np(a, b, atol=1e-5):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.shape != b.shape:
        return False, f'shape mismatch {a.shape} vs {b.shape}'
    diff = np.nanmax(np.abs(a - b)) if a.size else 0.0
    return diff <= atol, f'max_abs_diff={diff}'


def validate_scripted_launch_once(jax_step_fn, seed=1, num_agents=4):
    official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
    js = pack_official_states([official_state])
    obs = official_state[0].observation
    home = next(p for p in obs.planets if int(p[1]) == 0)
    # Use a tiny launch from player 0. This validates action processing, source debit,
    # production, fleet spawn, fleet movement, and planet rotation for one tick.
    actions = [[[int(home[0]), 0.0, 1]]] + [[] for _ in range(num_agents - 1)]
    for i, st in enumerate(official_state):
        st.action = actions[i]
    official_state = ow.interpreter(official_state, official_env)
    next_step = getattr(official_state[0].observation, 'step', 0) + 1
    for st in official_state:
        st.observation.step = next_step
    ja = pack_actions_for_jax([actions])
    js = jax_step_fn(js, ja, 6.0, 500)
    okp, msgp = compare_planets_np(np.asarray(js.planets[0]), pad_rows(official_state[0].observation.planets, MAX_PLANETS), atol=1e-4)
    okf, msgf = compare_fleets_np(np.asarray(js.fleets[0]), pad_rows(official_state[0].observation.fleets, MAX_FLEETS), atol=1e-4)
    if not okp:
        print('PLANET FAIL', msgp)
    if not okf:
        print('FLEET FAIL', msgf)
    if okp and okf:
        print('scripted launch validation passed')
    return bool(okp and okf)

# Chay sau cell jax_step:
# validate_scripted_launch_once(jax_step, seed=1, num_agents=4)


def make_official_state(planets, fleets, num_agents=2, step=1, angular_velocity=0.01, next_fleet_id=100):
    base = SimpleNamespace(
        observation=SimpleNamespace(
            step=step,
            planets=[list(x) for x in planets],
            fleets=[list(x) for x in fleets],
            next_fleet_id=next_fleet_id,
            angular_velocity=angular_velocity,
            initial_planets=[list(x) for x in planets],
            comets=[],
            comet_planet_ids=[],
        ),
        action=[],
        status='ACTIVE',
        reward=0,
    )
    return [base] + [SimpleNamespace(observation=SimpleNamespace(player=i), action=[], status='ACTIVE', reward=0) for i in range(1, num_agents)]


def run_case_compare(jax_step_fn, name, planets, fleets, num_agents=2, step=1, angular_velocity=0.01, ship_speed=6.0, episode_steps=500):
    official_state = make_official_state(planets, fleets, num_agents=num_agents, step=step, angular_velocity=angular_velocity)
    official_env = SimpleNamespace(configuration=SimpleNamespace(shipSpeed=ship_speed, episodeSteps=episode_steps, cometSpeed=4), done=False)
    js = pack_official_states([official_state])
    official_state = ow.interpreter(official_state, official_env)
    next_step = getattr(official_state[0].observation, 'step', 0) + 1
    for st in official_state:
        st.observation.step = next_step
    empty_actions = JaxAction(moves=jnp.zeros((1, MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32))
    js = jax_step_fn(js, empty_actions, float(ship_speed), int(episode_steps))
    okp, msgp = compare_planets_np(np.asarray(js.planets[0]), pad_rows(official_state[0].observation.planets, MAX_PLANETS), atol=1e-4)
    okf, msgf = compare_fleets_np(np.asarray(js.fleets[0]), pad_rows(official_state[0].observation.fleets, MAX_FLEETS), atol=1e-4)
    jr = np.asarray(js.rewards[0])[:num_agents]
    orr = np.asarray([float(s.reward) for s in official_state], dtype=np.float32)
    okr = np.max(np.abs(jr - orr)) <= 1e-5
    if not (okp and okf and okr):
        print('FAIL', name, 'planets', msgp, 'fleets', msgf, 'jax_rewards', jr, 'official_rewards', orr)
        return False
    return True


def validate_hard_cases(jax_step_fn):
    cases = [
        ('fleet_hits_sun', [[0,0,80,50,3,50,1]], [[0,0,60,50,math.pi,0,10]], 2, 1, 0.01, 6.0, 500),
        ('fleet_leaves_board', [[0,0,80,50,3,50,1]], [[0,0,99.5,50,0.0,0,10]], 2, 1, 0.01, 6.0, 500),
        ('fleet_survives', [[0,0,80,80,3,50,1]], [[0,0,50,30,0.0,0,10]], 2, 1, 0.01, 6.0, 500),
        ('fast_hit_before_board', [[0,1,98.0,50.0,2.0,50,1]], [[0,0,95.0,50.0,0.0,99,1000]], 2, 1, 0.01, 6.0, 500),
        ('fast_hit_before_sun', [[0,1,62.0,50.0,2.0,50,1]], [[0,0,65.0,50.0,math.pi,99,1000]], 2, 1, 0.0, 6.0, 500),
        ('combat_capture', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,30]], 2, 1, 0.01, 6.0, 500),
        ('combat_reinforce', [[0,0,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,25]], 2, 1, 0.01, 6.0, 500),
        ('combat_insufficient', [[0,-1,80,50,3,20,1]], [[0,0,76.0,50.0,0.0,99,5]], 2, 1, 0.01, 6.0, 500),
        ('two_attackers_capture', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,50],[1,1,76.0,50.0,0.0,99,30]], 2, 1, 0.01, 6.0, 500),
        ('two_attackers_tie', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,30],[1,1,76.0,50.0,0.0,99,30]], 2, 1, 0.01, 6.0, 500),
        ('winner_reinforces', [[0,0,80,50,3,15,1]], [[0,0,76.0,50.0,0.0,99,40],[1,1,76.0,50.0,0.0,99,25]], 2, 1, 0.01, 6.0, 500),
        ('winner_attacks_enemy', [[0,1,80,50,3,5,1]], [[0,0,76.0,50.0,0.0,99,50],[1,2,76.0,50.0,0.0,99,20]], 4, 1, 0.01, 6.0, 500),
        ('same_owner_sum', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,20],[1,0,76.0,50.0,0.0,99,15]], 2, 1, 0.01, 6.0, 500),
        ('reward_max_steps', [[0,0,80,80,3,50,1],[1,1,20,20,3,30,1]], [], 2, 498, 0.01, 6.0, 500),
        ('reward_tie', [[0,0,80,80,3,30,1],[1,1,20,20,3,30,1]], [], 2, 498, 0.01, 6.0, 500),
        ('reward_all_elim', [[0,-1,80,80,3,50,1]], [], 2, 1, 0.01, 6.0, 500),
        ('reward_4p_elim', [[0,2,80,80,3,40,1]], [], 4, 1, 0.01, 6.0, 500),
        ('reward_includes_fleet', [[0,0,80,80,3,10,1],[1,1,20,20,3,30,1]], [[0,0,50,30,0.0,0,50]], 2, 498, 0.01, 6.0, 500),
        ('rotating_swept_collision', [[0,-1,50.0,52.0,1.0,10,0]], [[0,0,49.0,50.0,0.0,1,1000]], 2, 1, math.pi, 2.0, 500),
    ]
    ok_all = True
    for case in cases:
        ok = run_case_compare(jax_step_fn, *case)
        print(case[0], 'OK' if ok else 'FAIL')
        ok_all = ok_all and ok
    print('hard cases', 'PASSED' if ok_all else 'FAILED')
    return ok_all

# Chay sau cell jax_step:
# validate_hard_cases(jax_step)



def canonical_live_rows(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[0] == 0:
        return arr
    live = arr[arr[:, 0] >= 0]
    if live.shape[0] == 0:
        return live
    order = np.argsort(live[:, 0], kind='stable')
    return live[order]


def compare_live_rows_np(a, b, name='rows', atol=1e-5):
    aa = canonical_live_rows(a)
    bb = canonical_live_rows(b)
    if aa.shape != bb.shape:
        return False, f'{name} live shape mismatch {aa.shape} vs {bb.shape}'
    diff = np.nanmax(np.abs(aa - bb)) if aa.size else 0.0
    return diff <= atol, f'{name} max_abs_diff={diff}'


def scripted_multi_step_actions(official_state, seed, turn, num_agents):
    # Deterministic pseudo-random legal-ish launches. This is only for simulator validation.
    rng = random.Random((int(seed) + 17) * 1000003 + int(turn) * 97 + int(num_agents))
    obs = official_state[0].observation
    actions = [[] for _ in range(num_agents)]
    for player_id in range(num_agents):
        owned = [p for p in obs.planets if int(p[1]) == player_id and int(p[5]) >= 4]
        owned.sort(key=lambda p: (-int(p[5]), -int(p[6]), int(p[0])))
        for p in owned[:2]:
            if rng.random() > 0.55:
                continue
            ships = max(1, min(int(p[5]) // 5, 8))
            angle = rng.random() * 2.0 * math.pi
            actions[player_id].append([int(p[0]), float(angle), int(ships)])
    return actions


def compare_step_state(js, official_state, num_agents, atol=1e-4):
    jp = np.asarray(js.planets[0])
    jf = np.asarray(js.fleets[0])
    op = pad_rows(official_state[0].observation.planets, MAX_PLANETS)
    of = pad_rows(official_state[0].observation.fleets, MAX_FLEETS)
    okp, msgp = compare_live_rows_np(jp, op, name='planets', atol=atol)
    okf, msgf = compare_live_rows_np(jf, of, name='fleets', atol=atol)
    jr = np.asarray(js.rewards[0])[:num_agents]
    orr = np.asarray([float(s.reward) for s in official_state], dtype=np.float32)
    okr = np.max(np.abs(jr - orr)) <= 1e-5
    return okp and okf and okr, (msgp, msgf, f'rewards jax={jr} official={orr}')


def validate_random_scripted_no_comet(jax_step_fn, seeds=(1, 2, 3), steps=40, num_agents=4):
    # Official first comet spawns at step 50, so keep this below that until comet support is ported.
    # This validates multi-turn launch, production, rotation, fleet movement, collisions, and rewards.
    if steps >= 49:
        raise ValueError('Use steps < 49 before comet support is implemented.')
    for seed in seeds:
        official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
        js = pack_official_states([official_state])
        ok, msg = compare_step_state(js, official_state, num_agents)
        if not ok:
            print('INITIAL FAIL seed', seed, msg)
            return False
        for turn in range(steps):
            actions = scripted_multi_step_actions(official_state, seed, turn, num_agents)
            for i, st in enumerate(official_state):
                st.action = actions[i]
            official_state = ow.interpreter(official_state, official_env)
            next_step = getattr(official_state[0].observation, 'step', 0) + 1
            for st in official_state:
                st.observation.step = next_step
            js = jax_step_fn(js, pack_actions_for_jax([actions]), 6.0, 500)
            ok, msg = compare_step_state(js, official_state, num_agents, atol=2e-4)
            if not ok:
                print('RANDOM SCRIPTED FAIL seed', seed, 'turn', turn, 'actions', actions)
                print(msg)
                return False
    print('random scripted no-comet validation passed')
    return True

# Chay sau cell jax_step:
# validate_random_scripted_no_comet(jax_step, seeds=(1, 2, 3), steps=40, num_agents=4)



def validate_random_scripted_with_comets(jax_step_fn, seeds=(1, 2, 3), steps=80, num_agents=4):
    # Same scripted actions as the no-comet test, but runs across the first comet spawn.
    for seed in seeds:
        official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
        js = pack_official_states([official_state])
        for turn in range(steps):
            actions = scripted_multi_step_actions(official_state, seed, turn, num_agents)
            for i, st in enumerate(official_state):
                st.action = actions[i]
            official_state = ow.interpreter(official_state, official_env)
            next_step = getattr(official_state[0].observation, 'step', 0) + 1
            for st in official_state:
                st.observation.step = next_step
            js = jax_step_fn(js, pack_actions_for_jax([actions]), 6.0, 500)
            ok, msg = compare_step_state(js, official_state, num_agents, atol=3e-4)
            if not ok:
                print('RANDOM COMET FAIL seed', seed, 'turn', turn, 'official_step', official_state[0].observation.step)
                print('actions', actions)
                print(msg)
                return False
    print('random scripted with-comets validation passed')
    return True

# Chay sau cell jax_step:
# validate_random_scripted_with_comets(jax_step, seeds=(1, 2, 3), steps=80, num_agents=4)


In [ ]:
# === AUTO VALIDATION RUN ===
# Cell nay tu chay khi Run all neu AUTO_RUN_VALIDATION=True.
# Neu co fail, notebook raise error de dung truoc khi train.


def run_all_validations():
    print('=== Orbit Wars JAX validation start ===')
    results = []

    print('[1/5] no-action + comet schedule')
    results.append(('noaction_60', validate_jax_step_against_official(jax_step, seeds=[1, 2, 3], steps=60, num_agents=4)))

    print('[2/5] scripted launch')
    results.append(('scripted_launch', validate_scripted_launch_once(jax_step, seed=1, num_agents=4)))

    print('[3/5] hard cases')
    results.append(('hard_cases', validate_hard_cases(jax_step)))

    print('[4/5] random scripted before comet')
    results.append(('random_no_comet_40', validate_random_scripted_no_comet(jax_step, seeds=(1, 2, 3), steps=40, num_agents=4)))

    print('[5/5] random scripted with comet')
    results.append(('random_with_comets_80', validate_random_scripted_with_comets(jax_step, seeds=(1, 2, 3), steps=80, num_agents=4)))

    print('=== Validation summary ===')
    ok_all = True
    for name, ok in results:
        print(f'{name}: {"PASS" if ok else "FAIL"}')
        ok_all = ok_all and bool(ok)
    print('OVERALL:', 'PASS' if ok_all else 'FAIL')
    if not ok_all:
        raise RuntimeError('JAX simulator validation failed. Do not train before fixing this.')
    return True


VALIDATION_OK = False
if AUTO_RUN_VALIDATION:
    VALIDATION_OK = run_all_validations()
else:
    print('AUTO_RUN_VALIDATION=False, skipped validation.')


In [ ]:
# === JAX PPO TRAINING CORE ===
# Run validation cells first. Policy learns to bias/select heuristic candidates, not raw actions.

class PolicyParams(NamedTuple):
    w1: jnp.ndarray
    b1: jnp.ndarray
    w2: jnp.ndarray
    b2: jnp.ndarray
    value_w1: jnp.ndarray
    value_b1: jnp.ndarray
    value_w2: jnp.ndarray
    value_b2: jnp.ndarray


class AdamState(NamedTuple):
    m: PolicyParams
    v: PolicyParams
    t: jnp.ndarray


def tree_zeros_like(tree):
    return jax.tree_util.tree_map(jnp.zeros_like, tree)


def init_policy_params(key):
    k1, k2, k3, k4 = jrandom.split(key, 4)
    return PolicyParams(
        w1=jrandom.normal(k1, (FEATURE_DIM, HIDDEN_DIM)) * 0.02,
        b1=jnp.zeros((HIDDEN_DIM,)),
        w2=jrandom.normal(k2, (HIDDEN_DIM, 1)) * 0.02,
        b2=jnp.zeros((1,)),
        value_w1=jrandom.normal(k3, (FEATURE_DIM, HIDDEN_DIM)) * 0.02,
        value_b1=jnp.zeros((HIDDEN_DIM,)),
        value_w2=jrandom.normal(k4, (HIDDEN_DIM, 1)) * 0.02,
        value_b2=jnp.zeros((1,)),
    )


def init_adam(params):
    return AdamState(m=tree_zeros_like(params), v=tree_zeros_like(params), t=jnp.asarray(0, dtype=jnp.int32))


def adam_update(params, grads, opt_state, lr=LEARNING_RATE, beta1=0.9, beta2=0.999, eps=1e-8):
    t = opt_state.t + 1
    m = jax.tree_util.tree_map(lambda m, g: beta1 * m + (1.0 - beta1) * g, opt_state.m, grads)
    v = jax.tree_util.tree_map(lambda v, g: beta2 * v + (1.0 - beta2) * (g * g), opt_state.v, grads)
    mhat = jax.tree_util.tree_map(lambda x: x / (1.0 - beta1 ** t), m)
    vhat = jax.tree_util.tree_map(lambda x: x / (1.0 - beta2 ** t), v)
    params = jax.tree_util.tree_map(lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + eps), params, mhat, vhat)
    return params, AdamState(m=m, v=v, t=t)


def make_initial_jax_batch(seed_start=0, batch_size=NUM_ENVS, prob_4p=TRAIN_PROB_4P):
    states = []
    rng = random.Random(int(seed_start))
    for i in range(batch_size):
        num_agents = 4 if rng.random() < prob_4p else 2
        seed = int(seed_start) * 100000 + i + 1
        states.append(official_initial_state(num_agents=num_agents, seed=seed)[0])
    return pack_official_states(states)


def candidate_features_for_player(planets, player_id):
    ids = planets[:, 0].astype(jnp.int32)
    owners = planets[:, 1].astype(jnp.int32)
    x = planets[:, 2]
    y = planets[:, 3]
    radius = planets[:, 4]
    ships = planets[:, 5]
    prod = planets[:, 6]
    alive = ids >= 0

    src_alive = alive[:, None]
    tgt_alive = alive[None, :]
    src_owned = src_alive & (owners[:, None] == player_id) & (ships[:, None] >= 4.0)
    not_same = ids[:, None] != ids[None, :]
    valid_pair = src_owned & tgt_alive & not_same

    dx = x[None, :] - x[:, None]
    dy = y[None, :] - y[:, None]
    dist = jnp.sqrt(dx * dx + dy * dy) + 1e-6
    target_neutral = owners[None, :] == -1
    target_enemy = (owners[None, :] >= 0) & (owners[None, :] != player_id)
    target_friend = owners[None, :] == player_id
    needed = jnp.where(target_friend, 1.0, ships[None, :] + 1.0)
    send_cap = jnp.maximum(1.0, jnp.floor(ships[:, None] * jnp.where(target_friend, 0.18, 0.55)))
    send = jnp.minimum(send_cap, jnp.maximum(1.0, needed + 2.0))
    valid_pair = valid_pair & (send > 0.0)

    score = (
        prod[None, :] * 4.0
        - ships[None, :] * 0.08
        - dist * 0.045
        + jnp.where(target_neutral, 7.0, 0.0)
        + jnp.where(target_enemy, 5.0, 0.0)
        - jnp.where(target_friend, 4.0, 0.0)
        + ships[:, None] * 0.015
    )
    score = jnp.where(valid_pair, score, -1.0e9)
    flat_score = score.reshape((-1,))
    top_score, top_idx = lax.top_k(flat_score, NUM_CANDIDATES)
    src_idx = top_idx // MAX_PLANETS
    tgt_idx = top_idx % MAX_PLANETS
    valid = top_score > -1.0e8

    sx = x[src_idx]
    sy = y[src_idx]
    tx = x[tgt_idx]
    ty = y[tgt_idx]
    angle = jnp.arctan2(ty - sy, tx - sx)
    action = jnp.stack([ids[src_idx].astype(jnp.float32), angle, send.reshape((-1,))[top_idx]], axis=-1)
    action = jnp.where(valid[:, None], action, jnp.zeros_like(action))

    feat = jnp.stack([
        ships[src_idx] / 200.0,
        prod[src_idx] / 8.0,
        radius[src_idx] / 8.0,
        ships[tgt_idx] / 200.0,
        prod[tgt_idx] / 8.0,
        radius[tgt_idx] / 8.0,
        dist[src_idx, tgt_idx] / 141.5,
        send.reshape((-1,))[top_idx] / 200.0,
        needed.reshape((-1,))[top_idx] / 200.0,
        target_neutral.reshape((-1,))[top_idx].astype(jnp.float32),
        target_enemy.reshape((-1,))[top_idx].astype(jnp.float32),
        target_friend.reshape((-1,))[top_idx].astype(jnp.float32),
        sx / 100.0,
        sy / 100.0,
        tx / 100.0,
        ty / 100.0,
        top_score / 50.0,
        jnp.ones((NUM_CANDIDATES,), dtype=jnp.float32),
    ], axis=-1)
    feat = jnp.where(valid[:, None], feat, jnp.zeros_like(feat))
    return action, feat, valid, top_score


def build_candidates_single(state: JaxState):
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    action, feat, valid, base_score = jax.vmap(lambda p: candidate_features_for_player(state.planets, p))(players)
    valid_player = players < state.player_count
    valid = valid & valid_player[:, None]
    return action, feat, valid, base_score


def policy_logits_values(params, features, valid, base_score=None):
    h = jnp.tanh(features @ params.w1 + params.b1)
    learned = (h @ params.w2 + params.b2).squeeze(-1) * BIAS_SCALE
    prior = 0.0 if base_score is None else base_score / 10.0
    logits = learned + prior
    logits = jnp.where(valid, logits, -1.0e9)
    any_valid = jnp.any(valid, axis=-1, keepdims=True)
    fallback = jnp.concatenate([jnp.zeros_like(logits[..., :1]), jnp.full_like(logits[..., 1:], -1.0e9)], axis=-1)
    logits = jnp.where(any_valid, logits, fallback)
    pooled = jnp.mean(features, axis=-2)
    vh = jnp.tanh(pooled @ params.value_w1 + params.value_b1)
    values = (vh @ params.value_w2 + params.value_b2).squeeze(-1)
    return logits, values


def sample_actions(params, state, key, temperature=TRAIN_TEMPERATURE):
    cand_action, features, valid, base_score = build_candidates_single(state)
    logits, values = policy_logits_values(params, features, valid, base_score)
    keys = jrandom.split(key, MAX_PLAYERS)
    chosen = jax.vmap(lambda k, lg: jrandom.categorical(k, lg / temperature))(keys, logits)
    chosen_action = jnp.take_along_axis(cand_action, chosen[:, None, None], axis=1).squeeze(1)
    has_valid = jnp.any(valid, axis=-1)
    chosen_action = jnp.where(has_valid[:, None], chosen_action, jnp.zeros_like(chosen_action))
    moves = jnp.zeros((MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32).at[:, 0, :].set(chosen_action)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    chosen_logp = jnp.take_along_axis(log_probs, chosen[:, None], axis=-1).squeeze(-1)
    probs = jax.nn.softmax(logits, axis=-1)
    entropy = -jnp.sum(probs * log_probs, axis=-1)
    return JaxAction(moves=moves), features, valid, base_score, chosen, chosen_logp, values, entropy


batched_sample_actions = jax.jit(jax.vmap(sample_actions, in_axes=(None, 0, 0, None)))


def rollout_once(params, state, key):
    def one_step(carry, _):
        st, k = carry
        k, ak = jrandom.split(k)
        action, features, valid, base_score, chosen, logp, value, entropy = batched_sample_actions(params, st, jrandom.split(ak, st.planets.shape[0]), TRAIN_TEMPERATURE)
        next_st = jax_step(st, action, 6.0, 500)
        reward = next_st.rewards
        done = next_st.done
        data = (features, valid, base_score, chosen, logp, value, reward, done, entropy)
        return (next_st, k), data
    (next_state, key), data = lax.scan(one_step, (state, key), None, length=ROLLOUT_STEPS)
    return next_state, key, data


def compute_gae(rewards, values, dones, last_values):
    def scan_fn(carry, x):
        gae, next_value = carry
        reward, value, done = x
        mask = 1.0 - done.astype(jnp.float32)[:, None]
        delta = reward + GAMMA * next_value * mask - value
        gae = delta + GAMMA * GAE_LAMBDA * mask * gae
        return (gae, value), gae
    (_, _), adv_rev = lax.scan(scan_fn, (jnp.zeros_like(last_values), last_values), (rewards[::-1], values[::-1], dones[::-1]))
    adv = adv_rev[::-1]
    returns = adv + values
    adv = (adv - adv.mean()) / (adv.std() + 1e-6)
    return adv, returns


def ppo_loss(params, batch):
    features, valid, base_score, chosen, old_logp, old_value, returns, advantages = batch
    logits, values = policy_logits_values(params, features, valid, base_score)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    new_logp = jnp.take_along_axis(log_probs, chosen[..., None], axis=-1).squeeze(-1)
    ratio = jnp.exp(new_logp - old_logp)
    unclipped = ratio * advantages
    clipped = jnp.clip(ratio, 1.0 - PPO_CLIP_EPS, 1.0 + PPO_CLIP_EPS) * advantages
    policy_loss = -jnp.mean(jnp.minimum(unclipped, clipped))
    value_loss = jnp.mean((returns - values) ** 2)
    probs = jax.nn.softmax(logits, axis=-1)
    entropy = -jnp.mean(jnp.sum(probs * log_probs, axis=-1))
    loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy
    return loss, (policy_loss, value_loss, entropy)


@jax.jit
def ppo_update(params, opt_state, batch):
    (loss, aux), grads = jax.value_and_grad(ppo_loss, has_aux=True)(params, batch)
    params, opt_state = adam_update(params, grads, opt_state)
    return params, opt_state, loss, aux


def train_ppo(seed=0, updates=TOTAL_PPO_UPDATES, num_envs=NUM_ENVS):
    key = jrandom.PRNGKey(seed)
    params = init_policy_params(key)
    opt_state = init_adam(params)
    state = make_initial_jax_batch(seed_start=seed + 100, batch_size=num_envs)
    for update in range(int(updates)):
        key, rk = jrandom.split(key)
        state, key, data = rollout_once(params, state, rk)
        features, valid, base_score, chosen, logp, values, rewards, dones, entropy = data
        _, _, _, _, _, _, last_values, _ = batched_sample_actions(params, state, jrandom.split(key, num_envs), TRAIN_TEMPERATURE)
        adv, ret = compute_gae(rewards, values, dones, last_values)
        batch = (features, valid, base_score, chosen, logp, values, ret, adv)
        params, opt_state, loss, aux = ppo_update(params, opt_state, batch)
        if (update + 1) % EVAL_EVERY_UPDATES == 0 or update == 0:
            mean_reward = np.asarray(rewards).sum(axis=0).mean(axis=0)
            print('update', update + 1, 'loss', float(loss), 'mean_player_reward', mean_reward)
        # Reset whole batch near episode horizon. This keeps code simple and deterministic.
        if ((update + 1) * ROLLOUT_STEPS) % 480 == 0:
            state = make_initial_jax_batch(seed_start=seed + 1000 + update, batch_size=num_envs)
    return params

print('JAX PPO training core ready. Run train_ppo(seed=0) after validation passes.')


In [ ]:
# === SAVE / LOAD TRAINED PARAMS ===
# Sau khi train: params = train_ppo(seed=0)
# Luu ra npz de co the gan vao submission runtime sau nay.


def save_policy_params(params, path=OUT_DIR / 'ppo_policy_params.npz'):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {name: np.asarray(getattr(params, name)) for name in params._fields}
    np.savez_compressed(path, **arrays)
    print('saved', path)
    return path


def load_policy_params(path=OUT_DIR / 'ppo_policy_params.npz'):
    data = np.load(Path(path))
    return PolicyParams(**{name: jnp.asarray(data[name]) for name in PolicyParams._fields})

# Example:
# params = train_ppo(seed=0)
# save_policy_params(params)


In [ ]:
# === OPTIONAL TRAIN RUN ===
# Mac dinh AUTO_START_TRAIN=False nen Run all chi validate, khong train dai.
# Khi muon train that, sua AUTO_START_TRAIN=True trong cell config roi Run all / run cell nay.

TRAINED_PARAMS = None
if AUTO_START_TRAIN:
    if AUTO_RUN_VALIDATION and not VALIDATION_OK:
        raise RuntimeError('Validation did not pass, training is blocked.')
    TRAINED_PARAMS = train_ppo(seed=0)
    save_policy_params(TRAINED_PARAMS)
else:
    print('AUTO_START_TRAIN=False, skipped long PPO train.')
    print('To train: set AUTO_START_TRAIN=True in config, then run all again.')
    print('Or run manually: params = train_ppo(seed=0); save_policy_params(params)')


## Ket luan thiet ke

Thiet ke JAX chuan phai di theo thu tu:

1. Port official engine `orbit_wars.py` sang JAX.
2. Validate tung step voi official engine.
3. Sau khi validate pass moi train PPO vectorized.
4. PPO hoc candidate bias tren heuristic, khong hoc raw action ngay tu dau.

Notebook nay da tao nen tang JAX-first. Phan con lai can code tiep la `jax_step` day du.
